In [ ]:
import os
import sounddevice as sd
import numpy as np
from faster_whisper import WhisperModel
import ollama
from IPython.display import clear_output
import time
from pydantic import BaseModel
import time


In [ ]:
# Test Audio Device is detected
print(sd.query_devices())

In [ ]:
# Initial Whisper Model Config
model_size = "small"
model = WhisperModel(model_size, device="cuda")

In [ ]:
#Fixed length recording audio
duration = 4.5  # seconds
fs = 16000
myrecording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()
print("stop")


In [ ]:
temp = [1,2,3,4,5,6,7]

In [ ]:
temp[:5]

In [ ]:
temp[-2:]

In [ ]:
chunks = 1 2 3 4 5 6 7 8 9
window size is 5
and overlap size is 2

when we gewt he window, we get
chunks[:5]
aka
1 2 3 4 5
and this audio, will have an overlap with the next set we want to do
teh next set we want to process is going to be
4 5 6 7 8

so we want to remove
1 2 3
so to that that we set chunks to be
chunks[3:]

In [ ]:
temp

In [ ]:
temp[3:]

In [ ]:
self._chunks = self._chunks[-(self.window_size - self.overlap_size):]


In [ ]:
chunks[-num_we_want_to_maintain:]

In [ ]:
sd.play(myrecording, fs)

In [ ]:
np.array([], dtype = np.float32)

AttributeError: 'tuple' object has no attribute 'get_speech_timestamps'

In [210]:

model_vad, utils = torch.hub.load(
    repo_or_dir= 'snakers4/silero-vad',
    model = 'silero_vad')

(get_speech_timestamps, _, _, _, _) = utils

audio_tensor = torch.from_numpy(myrecording.flatten()).float()

speech_timestamps = get_speech_timestamps(
  audio = audio_tensor,
  model = model_vad,
  return_seconds=True,  # Return speech timestamps in seconds (default is samples)
  min_speech_duration_ms=5,  # Minimum speech chunk duration)
  min_silence_duration_ms = 5
)

speech_timestamps

Using cache found in /home/evergreen/.cache/torch/hub/snakers4_silero-vad_master


[{'start': 0.5, 'end': 3.2}, {'start': 3.2, 'end': 3.6}]

In [ ]:
segments, info = model.transcribe(myrecording.flatten(), beam_size = 5, word_timestamps= True)

seg_list = []
for segment in segments:
    seg_list.append(segment)
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

In [311]:
audio_buffer = AudioBuffer(window_size = 5, overlap_size = 2, model_vad = model_vad, utils = utils)

In [236]:
len(window)

80000

In [312]:
data_temp = audio_buffer._find_cut_points(window)

[{'start': 0, 'end': 37344, 'len': 37344, 'c1_diff': 48000, 'c2_diff': 42656}, {'start': 37920, 'end': 41440, 'len': 3520, 'c1_diff': 10080, 'c2_diff': 38560}, {'start': 42528, 'end': 45536, 'len': 3008, 'c1_diff': 5472, 'c2_diff': 34464}, {'start': 50208, 'end': 57824, 'len': 7616, 'c1_diff': 2208, 'c2_diff': 22176}, {'start': 59424, 'end': 80000, 'len': 20576, 'c1_diff': 11424, 'c2_diff': 0}]


In [277]:
def calc_sil_len(x):
    x['len'] = x['end'] - x['start']
    return x

data_temp = [calc_sil_len(x) for x in data_temp]

In [314]:
min(data_temp, key = lambda x: x['c1_diff'])

{'start': 50208, 'end': 57824, 'len': 7616, 'c1_diff': 2208, 'c2_diff': 22176}

In [315]:
min(data_temp, key = lambda x: x['c2_diff'])

{'start': 59424, 'end': 80000, 'len': 20576, 'c1_diff': 11424, 'c2_diff': 0}

In [265]:
[calc_sil_len(x) for x in data_temp]

[{'start': 0, 'end': 37344, 'len': 37344},
 {'start': 37920, 'end': 41440, 'len': 3520},
 {'start': 42528, 'end': 45536, 'len': 3008},
 {'start': 50208, 'end': 57824, 'len': 7616},
 {'start': 59424, 'end': 80000, 'len': 20576}]

In [260]:
data_new

[{'start': 0, 'end': 37344, 'len': 37344},
 {'start': 37920, 'end': 41440, 'len': 3520},
 {'start': 42528, 'end': 45536, 'len': 3008},
 {'start': 50208, 'end': 57824, 'len': 7616},
 {'start': 59424, 'end': 80000, 'len': 20576}]

In [310]:
class AudioBuffer:
    def __init__(self, window_size = 5, overlap_size = 2, sample_rate = 16000,  model_vad = None, utils = None):
        self.window_size = window_size
        self.overlap_size = overlap_size
        self.sample_rate = sample_rate
        self.audio_data_buffer = np.array([], dtype = np.float32)
        self.window_size_samp = self.window_size * self.sample_rate
        self.overlap_size_samp = self.overlap_size * self.sample_rate
        self.model_vad = model_vad
        self.utils = utils
        
    def append_chunk(self, audio_data):
        #chunk preprocessing goes here, before append
        self.audio_data_buffer = np.append(self.audio_data_buffer, audio_data.copy())
        
    def get_buffer_size(self):
        return len(self.audio_data_buffer)

    def get_silence_timestamps(self, analysis_audio):
        
        (get_speech_timestamps, _, _, _, _) = utils
        
        speech_timestamps = get_speech_timestamps(
                  audio = analysis_audio,
                  model = self.model_vad,
                  return_seconds= False,  # Return speech timestamps in seconds (default is samples)
                  min_speech_duration_ms=5,  # Minimum speech chunk duration)
                  min_silence_duration_ms = 5
            )


        return speech_timestamps

    def _find_cut_points(self, analysis_audio):

        silence_data = self.get_silence_timestamps(analysis_audio)

        def calc_sil_len(x):
            x['len'] = x['end'] - x['start']
            return x

        silence_data = [calc_sil_len(x) for x in silence_data]

        def diff_to_cut(x):
            cut_1_target = self.window_size_samp - self.overlap_size_samp
            cut_2_target = self.window_size_samp

            x['c1_diff'] = abs(cut_1_target - x['start'])
            x['c2_diff'] = abs(cut_2_target - x['end'])
            return x

        silence_data = [diff_to_cut(x) for x in silence_data]
        print(silence_data)
        
        
        #best_cuts = {'c1' : c1_best, 'c2': c2_best}
        return silence_data
            
    def get_window(self):
        # return concatenated window if ready, else None
        if len(self.audio_data_buffer) >= self.window_size_samp:
            window = self.audio_data_buffer[:self.window_size_samp]
            self.audio_data_buffer = self.audio_data_buffer[(self.window_size_samp - self.overlap_size_samp):]
            return window
        return None

In [ ]:
class TranscriptionPipeline:
    def __init__(self, whisper_model, merge_selector):
        self.whisper = whisper_model
        self.merge_selector = merge_selector
        self.transcript_history = ""

    def process_audio(self, audio_window):
        segments, _ = self.whisper.transcribe(
            audio_window.flatten(),
            beam_size = 5)
        return " ".join(segment.text for segment in segments)

    def merge_with_history(self, new_transcript):
        #delegates merging
        self.transcript_history = self.merge_selector.merge(
            self.transcript_history, 
            new_transcript
        )
        return self.transcript_history

    def process_and_merge(self, audio_window):
        transcript = self.process_audio(audio_window)
        self.merge_with_history(transcript)

        return self.transcript_history
        
    

In [ ]:
example_string = "This is an example...string that I am writing's about now. yep"
example_string2 = "This is an example... string two that I am writing's about now. yep"

import string
import re
example_string.translate(str.maketrans('', '', string.punctuation)).lower()

out1 = re.sub(r'\.{3}(?=\S|$)', '... ', example_string)
out2 = re.sub(r'\.{3}(?=\S|$)', '... ', example_string2)

print(out1)
print(out2)

In [ ]:
class MergeStrategySelector:
    def __init__(self, context_words = 10):
        self.context_words = context_words
        self.stats = {'exact': 0, 'fuzzy': 0, 'llm': 0}
        self.simple_strategy = SimpleMergeStrategy()
        self.llm_strategy = LLMMergeStrategy(model_name = 'llama3.2:3b', context_words = self.context_words)

    def merge(self, existing_text, new_text):
        
        new_text_clean = re.sub(r'\.{3}(?=\S|$)', '... ', new_text) #this should be somewhere else, tbd
        
        if not existing_text:
            return new_text_clean

        context_text = " ".join(existing_text.split()[-self.context_words:])

        
        
        match_info = self._analyse_overlap(context_text, new_text_clean, max_match_len = 10)
        
        if match_info['type'] == "exact":
            merged_output = self.simple_strategy.merge(existing_text, new_text_clean, match_info)
            self.stats["exact"] += 1
        elif match_info['type'] == 'llm':
            merged_output = self.llm_strategy.merge(existing_text, new_text_clean, match_info)
            self.stats["llm"] += 1

        return merged_output

    def _analyse_overlap(self, context_text, new_text, max_match_len = 10):

        context_text_analysis = context_text.translate(str.maketrans('', '', string.punctuation)).lower()
        new_text_analysis = new_text.translate(str.maketrans('', '', string.punctuation)).lower()
        
        context_words = context_text_analysis.split()
        new_words = new_text_analysis.split()
        
        match_len_limit = min(len(context_words), len(new_words), max_match_len)
        
        match_index = 0
        for i in range(match_len_limit, 0,- 1):
            v1 = context_words[-i:]
            v2 = new_words[:i]
            if v1 == v2:
                match_index = i
                break
        merge_type = "exact"
        if match_index == 0:
            merge_type = "llm"
            print("DEBUG: Exact Merging Failed")
            print(context_words[-match_len_limit:])
            print(new_words[:match_len_limit])
            
        
        return {'type': merge_type, 'overlap_length': match_index}

In [ ]:
class SimpleMergeStrategy:
    def merge(self, existing_text, new_text, match_info):
        merged_list = existing_text.split() + new_text.split()[match_info['overlap_length']:]
        return " ".join(merged_list)

In [ ]:
class merged_response(BaseModel):
  merged_output: str 


class LLMMergeStrategy:
    def __init__(self, model_name, context_words = 10):
        self.model_name = model_name
        self.context_words = context_words

        
    def merge(self, existing_text, new_text, match_info):

        if not existing_text:
            return new_text
        
        context_fragment = " ".join(existing_text.split()[-self.context_words:])
        
        llm_message = '''Here are the two transcript fragments
        1: ''' + context_fragment + '''
        2: ''' + new_text


        
        response = ollama.chat(model=self.model_name, format = merged_response.model_json_schema(), messages=[
                {'role': 'system', 'content': '''You merge overlapping transcript fragments. The end of transcript 1 overlaps with the start of transcript 2. Remove the duplication and return ONLY the merged text.
                
                Rules:
                - Output ONLY the merged text
                - No explanations, notes, or commentary
                - If there is no clear overlap, return transcript 2 unchanged
                
                Example:
                1: The cat sat on the mat
                2: on the mat and then fell asleep
                Output: The cat sat on the mat and then fell asleep'''}
                ,
                {'role': 'user', 'content': llm_message}
            ])
    
        output_text = existing_text[:-len(context_fragment)] + " " + merged_response.model_validate_json(response.message.content).merged_output
    
        return output_text

In [ ]:
whisper_model = WhisperModel(model_size, device="cuda")
audio_buffer = AudioBuffer(window_size = 5, overlap_size = 2)
merge_selector = MergeStrategySelector(context_words = 10)
pipeline = TranscriptionPipeline(whisper_model, merge_selector)

def audio_callback(indata, frames, times, status):
    audio_buffer.append_chunk(indata)

with sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000):
    while True:
        sd.sleep(100)
        window = audio_buffer.get_window()
        if window is None:
            continue
        transcribed_text = pipeline.process_and_merge(window)
        print(audio_buffer.get_buffer_size())
        print(transcribed_text)

In [ ]:
merge_selector.stats

In [ ]:
class merged_response(BaseModel):
  merged_output: str 

def text_output():
    #clear_output()

    global text_string
    global transcript_iter
    
    print("Transcript_iter - " + str(transcript_iter)) 
    print("----")
    print("text_string initially is: \n" + text_string)
    
    last_10_words = " ".join(text_string.split()[-10:])
    text_string = text_string[:-len(last_10_words)]

    print("----")
    print("last_10_words is: \n" + last_10_words)
    print("----")
    print("text_string with words removed: \n" + text_string)
    print("----")
    
    
    section_a = last_10_words
    section_b = whisper_out

    print("whisper_out is: \n" + whisper_out)
    print("----")
    llm_message = '''Here are the two transcript fragments
    1: ''' + section_a + '''
    2: ''' + section_b

    start = time.time()
    response = ollama.chat(model='llama3.2:3b', format = merged_response.model_json_schema(), messages=[
        {'role': 'system', 'content': '''You merge overlapping transcript fragments. The end of transcript 1 overlaps with the start of transcript 2. Remove the duplication and return ONLY the merged text.
        
        Rules:
        - Output ONLY the merged text
        - No explanations, notes, or commentary
        - If there is no clear overlap, return transcript 2 unchanged
        
        Example:
        1: The cat sat on the mat
        2: on the mat and then fell asleep
        Output: The cat sat on the mat and then fell asleep'''}
        ,
        {'role': 'user', 'content': llm_message}
    ])

    llm_time = time.time() - start
    print(f"LLM took: {llm_time:.3f}s")

    text_string += " " + merged_response.model_validate_json(response.message.content).merged_output
    transcript_iter += 1

    print("Updated text_string is: \n" + text_string)
    
    


In [ ]:
test_audio = 
segments, info = model.transcribe(full_audio.flatten(), beam_size = 5)
for segment in segments:
            whisper_out += segment.text

In [ ]:
text_string = ""
transcript_iter = 1

audio_chunks = []
def audio_callback(indata, frames, time, status):
    audio_chunks.append(indata.copy())

In [ ]:
stream = sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000)

In [ ]:
stream.start()
while True:
    if len(audio_chunks) >= 5:
        full_audio = np.concatenate(audio_chunks)
        del audio_chunks[0:3]
        start = time.time()
        segments, info = model.transcribe(full_audio.flatten(), beam_size = 5)
        whisper_out = ""
        for segment in segments:
            whisper_out += segment.text
        whisper_time = time.time() - start
        print(f"Whisper took: {whisper_time:.3f}s")
        text_output()
    sd.sleep(100)

In [ ]:
stream.stop()

In [ ]:
stream = sd.InputStream(samplerate = 16000, channels = 1, callback = audio_callback, blocksize=16000)

In [ ]:
stream.start()

In [ ]:
#playing with vad

import torch
import torchaudio

from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
torch.set_num_threads(1)


In [ ]:

model_vad, utils = torch.hub.load(
    repo_or_dir= 'snakers4/silero-vad',
    model = 'silero_vad')

(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

audio_tensor = torch.from_numpy(myrecording.flatten()).float()

speech_timestamps = get_speech_timestamps(
  audio = audio_tensor,
  model = model_vad,
  return_seconds=True,  # Return speech timestamps in seconds (default is samples)
  min_speech_duration_ms=5,  # Minimum speech chunk duration)
  min_silence_duration_ms = 5
)

speech_timestamps

In [ ]:
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

In [ ]:
get_speech_timestamps

In [ ]:
audio_tensor = torch.from_numpy(myrecording.flatten()).float()

In [ ]:
sd.play(myrecording, fs)

In [ ]:
speech_timestamps = get_speech_timestamps(
  audio = audio_tensor,
  model = model_vad,
  return_seconds=True,  # Return speech timestamps in seconds (default is samples)
  min_speech_duration_ms=5,  # Minimum speech chunk duration)
  min_silence_duration_ms = 5
)
    

speech_timestamps